# Figure 6 — Score-operator tests (S10, S18, S18b, S22, S18d)

Sources: `benchmarks/synthetic_multimodal/s10/operators.csv` (score-J vs OT flow-J fork AUC),
`benchmarks/perturb_k562/capture.csv` + `effect_tests.json` (S18: fraction of each knockdown's
shift captured by m soft directions), `benchmarks/perturb_k562_occupancy/{h1.csv,basins.csv,h2.json}`
(S18b: H1 control-state re-weighting; H2 basin stiffness vs occupancy sensitivity),
`benchmarks/real_flow/s22/scores.csv` + `summary.json` (S22: clone λ₁ vs LARRY sister-cell fate entropy, seeds 0–2, with
density and centroid-distance baselines) and `benchmarks/perturb_dose/gates.json` + `capture_{rpe1,k562}_{N1,N2}.csv`
(S18d: weak-tertile capture of T-target displacements by score-J and covariance directions, shared-basis null95).

In [1]:
from _style import *
import json as _json, re
from scipy.stats import rankdata, spearmanr

prov = Provenance("Fig6", notebook="notebooks/fig6_score_operator.ipynb")
ops = pd.read_csv(prov.use("benchmarks/synthetic_multimodal/s10/operators.csv", "a", "S10"))
cap = pd.read_csv(prov.use("benchmarks/perturb_k562/capture.csv", "b", "S18"))
eff = _json.loads(prov.use("benchmarks/perturb_k562/effect_tests.json", "b", "S18").read_text())
h1 = pd.read_csv(prov.use("benchmarks/perturb_k562_occupancy/h1.csv", "c", "S18b"))
bas = pd.read_csv(prov.use("benchmarks/perturb_k562_occupancy/basins.csv", "c", "S18b"))
h2 = _json.loads(prov.use("benchmarks/perturb_k562_occupancy/h2.json", "c", "S18b").read_text())
V = prov.value
s22 = pd.read_csv(prov.use("benchmarks/real_flow/s22/scores.csv", "d", "S22"))
s22sum = _json.loads(prov.use("benchmarks/real_flow/s22/summary.json", "d", "S22").read_text())
dgates = _json.loads(prov.use("benchmarks/perturb_dose/gates.json", "d", "S18d").read_text())
dcap = {(s_, n_): pd.read_csv(prov.use(f"benchmarks/perturb_dose/capture_{s_}_{n_}.csv", "d", "S18d"))
        for s_ in ["rpe1", "k562"] for n_ in ["N1", "N2"]}

fig = plt.figure(figsize=(DOUBLE_COL, 4.8))
gs = fig.add_gridspec(2, 4, hspace=0.55, wspace=0.7)
gsb = gs[1, :].subgridspec(1, 5, wspace=0.8, width_ratios=[1, 1, 0.12, 1.1, 1.3])

In [2]:
ax = fig.add_subplot(gs[0, 0:2])
presets = ["v3_default", "v3_stress", "v3_batch", "v3_transition"]
o = ops[(ops.method == "pca") & (ops.truth_class == "saddle")]
for i, pr in enumerate(presets):
    for op, col, dx, m_ in [("J_score", C_SCORE, -0.12, "s"), ("J_flow_ot", C_FLOW, 0.12, "o")]:
        v = o[(o.preset == pr) & (o.operator == op)].sort_values("seed").auc_saddle_vs_committed
        ax.scatter(i + dx + np.array([-0.04, 0, 0.04]), v, color=col, marker=m_, s=12, zorder=3)
        V(f"a_{op}_{pr}_min", v.min()); V(f"a_{op}_{pr}_max", v.max())
sj = o[(o.operator == "J_score") & o.preset.isin(["v3_default", "v3_stress", "v3_batch"])].auc_saddle_vs_committed
V("a_score_dev_presets_min", sj.min()); V("a_score_dev_presets_max", sj.max())
V("a_score_dev_n_below_0_45", int((sj < 0.45).sum())); V("a_score_dev_n", int(len(sj)))
ax.axhline(0.5, color=C_BASE, lw=0.5, ls=":")
ax.set_xticks(range(4)); ax.set_xticklabels([p.replace("v3_", "") for p in presets])
ax.set_ylabel("fork AUC (saddle vs committed)"); ax.set_ylim(0, 1.05)
ax.legend(handles=[plt.Line2D([], [], marker="s", ls="", color=C_SCORE, ms=3.5, label="score J"),
                   plt.Line2D([], [], marker="o", ls="", color=C_FLOW, ms=3.5, label="OT flow J")], fontsize=5.5, loc="center left", bbox_to_anchor=(0.0, 0.62))
ax.set_title("S10: non-stationary v3 presets (PCA-20)", fontsize=6.5)
panel_letter(ax, "a", x=-0.13)
prov.panel("a", "S10 fork AUC of score J vs OT flow J on four non-stationary v3 presets (3 seeds each)")

In [3]:
ax = fig.add_subplot(gs[0, 2:4])
c = cap[(cap.k == 20) & (cap.m == 3)]
meths = [("local_score_J", "local\nscore J", C_SCORE), ("global_score_J", "global\nscore J", C_SCORE),
         ("local_covariance", "local\ncovariance", C_BASE), ("global_covariance", "global\ncovariance", C_BASE)]
parts = ax.violinplot([c[m].values for m, _, _ in meths], positions=range(4), widths=0.75, showmedians=True, showextrema=False)
for pc, (_, _, col) in zip(parts["bodies"], meths):
    pc.set_facecolor(col); pc.set_alpha(0.45); pc.set_edgecolor("none")
parts["cmedians"].set_color("black")
chance = float(c.random_expectation.iloc[0])
ax.axhline(chance, color="black", ls="--", lw=0.7)
ax.text(3.48, chance - 0.02, f"chance {f2(chance)}", fontsize=6, ha="right", va="top")
for i, (m, _, _) in enumerate(meths):
    med = c[m].median(); V(f"b_median_{m}", med)
    ax.text(i, 1.02, f2(med, 3), ha="center", fontsize=6)
ax.set_xticks(range(4)); ax.set_xticklabels([l for _, l, _ in meths], fontsize=6)
ax.set_ylabel("shift fraction captured\n(3 directions, PCA-20)"); ax.set_ylim(0, 1.08)
V("b_n_targets", int(c.target.nunique())); V("b_n_eligible", eff["n_targets_eligible"]); V("b_n_significant", eff["n_targets_significant"])
V("b_P1_fraction_above_null95", float(c.local_score_exceeds_null95.mean())); V("b_chance", chance)
V("b_random_null_mean_median", float(c.random_null_mean.median()))
for k in [10, 15, 30]:
    ck = cap[(cap.k == k) & (cap.m == 3)]
    V(f"b_k{k}_median_local_score_J", ck.local_score_J.median()); V(f"b_k{k}_median_global_covariance", ck.global_covariance.median())
ax.set_title(f"S18: K562 CRISPRi, {c.target.nunique()} significant knockdowns", fontsize=6.5)
panel_letter(ax, "b", x=-0.3)
prov.panel("b", "S18 capture fraction per knockdown (k=20, m=3) for score-J and covariance directions; dashed = random-subspace expectation")

In [4]:
ax = fig.add_subplot(gsb[0, 0])
hh = h1[h1.k == 20]
ax.hist(hh.explained_fraction, bins=30, color=C_BASE)
ax.axvline(0.5, color="black", ls="--", lw=0.7); ax.axvline(hh.explained_fraction.median(), color=C_SCORE, lw=1)
ax.set_xlabel("H1 explained fraction\n(re-weighting control states)"); ax.set_ylabel("knockdowns")
V("c_H1_median", hh.explained_fraction.median()); V("c_H1_frac_above_noise95", hh.exceeds_noise_p95.astype(str).eq("True").mean())
V("c_H1_n_above_noise95", int(hh.exceeds_noise_p95.astype(str).eq("True").sum())); V("c_H1_n", int(len(hh)))
ax.set_title("S18b H1", fontsize=6.5)
panel_letter(ax, "c", x=-0.4)

ax = fig.add_subplot(gsb[0, 1])
sb = bas[(bas.k == 20) & (bas.scheme == "score_J")]
ax.scatter(sb.stiffness, sb.sensitivity_median_abs_log2_fc, s=9, color=C_SCORE, lw=0)
ax.set_xlabel("basin stiffness (score J)"); ax.set_ylabel("occupancy sensitivity\n(median |log2 FC|)")
# recompute the partial Spearman (ranks residualised on intercept, basin-size rank, covariance-trace rank)
X = np.c_[np.ones(len(sb)), rankdata(sb.n_controls), rankdata(sb.spread_trace_covariance)]
res = lambda y: (lambda r: r - X @ np.linalg.lstsq(X, r, rcond=None)[0])(rankdata(y))
rho_p = float(np.corrcoef(res(sb.stiffness), res(sb.sensitivity_median_abs_log2_fc))[0, 1])
rho_o = spearmanr(sb.stiffness, sb.sensitivity_median_abs_log2_fc).statistic
j20 = h2["by_k"]["20"]
assert abs(rho_p - j20["stiffness_partial"]["rho"]) < 1e-9 and abs(rho_o - j20["stiffness_ordinary"]["rho"]) < 1e-9
V("c_H2_partial_rho", rho_p); V("c_H2_partial_p_negative", j20["stiffness_partial"]["p_one_sided_negative"])
V("c_H2_ordinary_rho", rho_o); V("c_n_basins", int(len(sb)))
lb = bas[(bas.k == 20) & (bas.scheme == "leiden")]
V("c_leiden_ordinary_rho", spearmanr(1 / lb.spread_trace_covariance, lb.sensitivity_median_abs_log2_fc).statistic)
V("c_leiden_ordinary_p_negative", j20["leiden_ordinary"]["p_one_sided_negative"])
ax.set_title(f"S18b H2: partial ρ {f2(rho_p)},\none-sided p {f2(j20['stiffness_partial']['p_one_sided_negative'])}", fontsize=6.2)
prov.panel("c", "S18b: H1 explained fraction per knockdown (k=20); H2 basin stiffness vs occupancy sensitivity (59 score-J basins)")

In [5]:
# d (left): S22 — Spearman rho with next-time fate entropy H, seeds 0-2 (seed 0 gated, filled)
ax = fig.add_subplot(gsb[0, 3])
cohorts = [("pooled", "pooled"), ("undifferentiated", "Undiff.")]
series = [("rho_lambda1_H", "λ₁ (score J)", C_SCORE, "s"), ("rho_density_H", "density", C_BASE, "o"),
          ("rho_centroid_distance_H", "centroid dist.", "black", "^")]
for i, (coh, _) in enumerate(cohorts):
    for j, (col_, _, colr, mk) in enumerate(series):
        x0 = i + (j - 1) * 0.27
        for sd in [0, 1, 2]:
            r_ = s22[(s22.cohort == coh) & (s22.seed == sd)].iloc[0]
            ax.scatter(x0 + (sd - 1) * 0.06, r_[col_], marker=mk, s=11, zorder=3,
                       facecolor=colr if sd == 0 else "white", edgecolor=colr, linewidths=0.6)
            V(f"d_S22_{coh}_seed{sd}_{col_}", float(r_[col_]))
        if col_ == "rho_lambda1_H":
            V(f"d_S22_{coh}_seed0_p", float(s22[(s22.cohort == coh) & (s22.seed == 0)].p_permutation_positive.iloc[0]))
            V(f"d_S22_{coh}_seed0_partial", float(s22[(s22.cohort == coh) & (s22.seed == 0)].partial_rho_given_density.iloc[0]))
            V(f"d_S22_{coh}_n_pairs", int(s22[(s22.cohort == coh) & (s22.seed == 0)].n_pairs.iloc[0]))
ax.axhline(0, color=C_BASE, lw=0.6, ls=":")
ax.set_xticks(range(len(cohorts))); ax.set_xticklabels([l for _, l in cohorts]); ax.set_xlim(-0.55, 1.55)
lam = s22[s22.cohort.isin(["pooled", "undifferentiated"])]
V("d_S22_lambda_min_all_seeds", float(lam.rho_lambda1_H.min())); V("d_S22_lambda_max_all_seeds", float(lam.rho_lambda1_H.max()))
V("d_S22_n_tied", int(s22[(s22.cohort == "pooled") & (s22.seed == 0)].n_tied_pairs.iloc[0]))
V("d_S22_n_unique_clones", int(s22[(s22.cohort == "pooled") & (s22.seed == 0)].n_unique_clones.iloc[0]))
gates22 = {"L1": s22sum["gate_L1_pooled"]["passed"], "L2": s22sum["gate_L2_undifferentiated"]["passed"], "L3": s22sum["gate_L3_baselines"]["passed"]}
V("d_S22_gates", gates22)
ax.set_ylim(-0.2, 0.3)
ax.set_ylabel("Spearman ρ with sister-cell\nfate entropy H")
ax.legend(handles=[plt.Line2D([], [], marker=mk, ls="", color=colr, ms=3.5, label=lb) for _, lb, colr, mk in series]
          + [plt.Line2D([], [], marker="o", ls="", mfc="white", mec="black", ms=3.5, label="seeds 1–2 (report only)")],
          fontsize=5.5, loc="upper center", bbox_to_anchor=(0.45, -0.15), ncol=2, handletextpad=0.2, columnspacing=0.6, borderaxespad=0.2, labelspacing=0.25)
ax.set_title("S22: LARRY, L1–L3 " + ("fail" if not any(gates22.values()) else "see legend"), fontsize=6.2)
panel_letter(ax, "d")

# d (right): S18d — weak-tertile capture of T-target displacements, per target; median bars; shared-basis null95
ax = fig.add_subplot(gsb[0, 4])
cats = [("rpe1", "N1"), ("rpe1", "N2"), ("k562", "N1"), ("k562", "N2")]
for i, (s_, n_) in enumerate(cats):
    e = dgates["comparisons"][s_][n_]["weak"]
    d_ = dcap[(s_, n_)]; t_ = d_[(d_.tertile == "weak") & d_.group_T]
    assert len(t_) == e["n_T"] and abs(t_.capture_S_J.median() - e["median_S_J_T"]) < 1e-12
    for col_, colr, dx, mk, key in [("capture_S_J", C_SCORE, -0.18, "s", "median_S_J_T"), ("capture_S_C", C_BASE, 0.18, "o", "median_S_C_T")]:
        jit = np.linspace(-0.06, 0.06, len(t_)) if len(t_) > 1 else np.zeros(1)
        ax.scatter(i + dx + jit, t_[col_].values, marker=mk, s=6, color=colr, alpha=0.7, lw=0, zorder=3)
        ax.plot([i + dx - 0.13, i + dx + 0.13], [e[key]] * 2, color=colr, lw=1.4, zorder=4)
        V(f"d_S18d_{s_}_{n_}_{key}", e[key])
    ax.plot([i - 0.38, i + 0.38], [e["null_across_T_median_p95"]] * 2, color="black", lw=0.7, ls="--", zorder=2)
    V(f"d_S18d_{s_}_{n_}_null95", e["null_across_T_median_p95"])
    V(f"d_S18d_{s_}_{n_}_p_wilcoxon", e["wilcoxon"]["p_one_sided_greater"]); V(f"d_S18d_{s_}_{n_}_p_mw", e["mannwhitney"]["p_one_sided_greater"])
    V(f"d_S18d_{s_}_{n_}_D", [e["D1_criterion_met"], e["D2_criterion_met"], e["D3_criterion_met"]])
for k_ in ["rpe1", "k562"]:
    V(f"d_S18d_n_T_{k_}", dgates["n_T"][k_]); V(f"d_S18d_n_M_{k_}", dgates["n_M"][k_])
_nd = {_json.loads(prov.use(f"benchmarks/perturb_dose/capture_{s_}_{n_}.json", "d", "S18d").read_text())["null_definition"] for s_, n_ in cats}
assert len(_nd) == 1; V("d_S18d_n_null", int(re.match(r"(\d+) across-T medians", _nd.pop()).group(1)))
V("d_S18d_outcome", dgates["outcome"]); V("d_S18d_threshold", dgates["underpowered_threshold_n_T"])
ax.axvline(1.5, color=C_BASE, lw=0.5)
ax.set_ylim(0, 1.36); ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.text(0.5, 1.33, f"RPE1\n{dgates['outcome'].replace('underpowered', 'under-\npowered')}\n(n_T = {dgates['n_T']['rpe1']})", ha="center", va="top", fontsize=6)
ax.text(2.5, 1.33, f"K562\nreport\nonly\n(n_T = {dgates['n_T']['k562']})", ha="center", va="top", fontsize=6)
ax.set_xticks(range(4)); ax.set_xticklabels([n_ for _, n_ in cats]); ax.set_xlim(-0.55, 3.55)
ax.set_ylabel("weak-tertile shift captured\n(T targets, 3 directions)")
ax.legend(handles=[plt.Line2D([], [], marker="s", ls="-", lw=1.2, color=C_SCORE, ms=3, label="score J"),
                   plt.Line2D([], [], marker="o", ls="-", lw=1.2, color=C_BASE, ms=3, label="covariance"),
                   plt.Line2D([], [], ls="--", lw=0.7, color="black", label="null95")],
          fontsize=5.5, loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, handletextpad=0.3, columnspacing=0.6, borderaxespad=0.2, handlelength=1.4)
ax.set_title("S18d: weak tertile", fontsize=6.2)
prov.panel("d", "S22 Spearman rho of clone lambda1 / density / centroid distance with LARRY next-time fate entropy (pooled, Undifferentiated; seeds 0-2); "
                "S18d weak-tertile capture of T-target displacements by score-J vs covariance directions with shared-basis null95 (RPE1 underpowered, K562 report only)")
save(fig, "Fig6")

[save] main/Fig6: 163.6 x 111.7 mm, PNG 3865 x 2639 px @ 600 dpi


(PosixPath('/Users/terooatt/Documents/scJDO2-orientation/manuscript/figures/main/Fig6.pdf'),
 PosixPath('/Users/terooatt/Documents/scJDO2-orientation/manuscript/figures/main/Fig6.png'))

In [6]:
v = prov.values
R = lambda a, b: rng_str(v[a], v[b])
M = lambda x, nd=2: f2(x, nd).replace('-', '−')
P_ = lambda p: f2(p, 3) if p < 0.995 else f2(p, 0)
PF = lambda b: 'failed' if not b else 'passed'
prov.legend(f"""
**Figure 6 | The score operator: a density-curvature descriptor, not validated against perturbation or lineage (S10, S18, S18b, S22, S18d).**
**a**, S10: fork AUC of score J (−β∇²E, energy network fitted by exact-Laplacian score matching) vs OT flow J on four non-stationary v3
presets (PCA-20, 3 seeds each). Score-J AUC on default {R('a_J_score_v3_default_min', 'a_J_score_v3_default_max')}, stress
{R('a_J_score_v3_stress_min', 'a_J_score_v3_stress_max')} and batch {R('a_J_score_v3_batch_min', 'a_J_score_v3_batch_max')} (AUC < 0.45, i.e. committed cells ranked above saddle cells, on
{v['a_score_dev_n_below_0_45']} of {v['a_score_dev_n']} seeds), {R('a_J_score_v3_transition_min', 'a_J_score_v3_transition_max')} on transition;
OT flow J {R('a_J_flow_ot_v3_default_min', 'a_J_flow_ot_v3_default_max')} on default.
**b**, S18 (pre-registered; P1 failed): Replogle K562 CRISPRi; all fits on 10,691 non-targeting controls; {v['b_n_significant']} of {v['b_n_eligible']}
targets with ≥ 100 cells passed the BH-adjusted permutation test. Fraction of each knockdown's centroid shift captured by m = 3 directions
in control PCA-20: local score-J soft axes {f2(v['b_median_local_score_J'], 3)}, global score J {f2(v['b_median_global_score_J'], 3)}, local covariance
{f2(v['b_median_local_covariance'], 3)}, global covariance {f2(v['b_median_global_covariance'], 3)} (medians); random-subspace expectation {f2(v['b_chance'])}.
P1 (fraction of targets above their random-null 95th percentile ≥ 0.70): {f2(v['b_P1_fraction_above_null95'], 3)}. k = 10/15/30 (report only): local score J
{f2(v['b_k10_median_local_score_J'], 3)}/{f2(v['b_k15_median_local_score_J'], 3)}/{f2(v['b_k30_median_local_score_J'], 3)}.
**c**, S18b (pre-registered; both gates failed). Left, H1: fraction of each knockdown's energy distance explained by re-weighting control states
(k = 20): median {f2(v['c_H1_median'])} (required ≥ 0.5); {v['c_H1_n_above_noise95']} of {v['c_H1_n']} targets ({f2(v['c_H1_frac_above_noise95'])}) above the size-matched noise 95th percentile.
Right, H2: score-J basin stiffness vs occupancy sensitivity across {v['c_n_basins']} basins; partial Spearman ρ (ranks residualised on basin size and
covariance trace) {f2(v['c_H2_partial_rho'])}, one-sided p (negative) {f2(v['c_H2_partial_p_negative'])} (10,000 permutations; recomputed ρ matches h2.json);
ordinary ρ {f2(v['c_H2_ordinary_rho'])}; operator-free Leiden spread ρ {f2(v['c_leiden_ordinary_rho'])} (p⁻ {f2(v['c_leiden_ordinary_p_negative'])}).
**d**, Left, S22 (pre-registered; L1 {PF(v['d_S22_gates']['L1'])}, L2 {PF(v['d_S22_gates']['L2'])}, L3 {PF(v['d_S22_gates']['L3'])}): Spearman ρ between a LARRY clone's median
score-J λ₁ on day *t* and the entropy H of its sister cells' types on day *t* + 2 (filled: seed 0, gated; open: seeds 1–2, report only), next to the
local-density and distance-to-Undifferentiated-centroid baselines. Pooled ({v['d_S22_pooled_n_pairs']:,} clone/time pairs, {v['d_S22_n_tied']} with tied modal types): λ₁ ρ
{M(v['d_S22_pooled_seed0_rho_lambda1_H'], 3)} (one-sided permutation p {P_(v['d_S22_pooled_seed0_p'])}); Undifferentiated ({v['d_S22_undifferentiated_n_pairs']:,} pairs): λ₁ ρ {M(v['d_S22_undifferentiated_seed0_rho_lambda1_H'])}
(p {P_(v['d_S22_undifferentiated_seed0_p'])}), centroid distance {M(v['d_S22_undifferentiated_seed0_rho_centroid_distance_H'])}, density {M(v['d_S22_undifferentiated_seed0_rho_density_H'])}; partial ρ of λ₁ given density
{M(v['d_S22_undifferentiated_seed0_partial'])}. λ₁ ρ over seeds 0–2 and both cohorts: {M(v['d_S22_lambda_min_all_seeds'], 3)} to {M(v['d_S22_lambda_max_all_seeds'], 3)}.
Right, S18d: fraction of the weak-knockdown-tertile centroid shift of each transcription-machinery (T) target captured by three control score-J soft directions
(vermillion) or three control-covariance directions (gray); bars, medians over T targets; dashed, 95th percentile of the across-T median under {v['d_S18d_n_null']:,}
covariance-matched random subspaces (shared basis). N1, depth-projected log-normalization; N2, analytic Pearson residuals. RPE1 (pre-registered; {v['d_S18d_outcome']}:
{v['d_S18d_n_T_rpe1']} qualifying T target, ≥ {v['d_S18d_threshold']} required): score J {M(v['d_S18d_rpe1_N1_median_S_J_T'])}/{M(v['d_S18d_rpe1_N2_median_S_J_T'])} (N1/N2), covariance
{M(v['d_S18d_rpe1_N1_median_S_C_T'])}/{M(v['d_S18d_rpe1_N2_median_S_C_T'])}, null95 {M(v['d_S18d_rpe1_N1_null95'])}/{M(v['d_S18d_rpe1_N2_null95'])}. K562 (report only; {v['d_S18d_n_T_k562']} T, {v['d_S18d_n_M_k562']} M targets): score J
{M(v['d_S18d_k562_N1_median_S_J_T'])}/{M(v['d_S18d_k562_N2_median_S_J_T'])}, covariance {M(v['d_S18d_k562_N1_median_S_C_T'])}/{M(v['d_S18d_k562_N2_median_S_C_T'])}, null95 {M(v['d_S18d_k562_N1_null95'])}/{M(v['d_S18d_k562_N2_null95'])}; score J above
covariance, one-sided paired Wilcoxon p {M(v['d_S18d_k562_N1_p_wilcoxon'])}/{M(v['d_S18d_k562_N2_p_wilcoxon'])}; T above mitochondrial/translation (M) targets, one-sided Mann–Whitney
p {M(v['d_S18d_k562_N1_p_mw'], 3)}/{M(v['d_S18d_k562_N2_p_mw'], 3)}. Supplementary Tables S2 and S3.
""")
prov.write()